# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 28.16 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [23]:
# YOUR CHANGES HERE
import json
import pandas as pd

with open("SQuAD-explorer/dataset/train-v1.1.json", encoding="utf-8") as f:
    squad = json.load(f)

rows = []
for document in squad["data"]:
    for paragraph_index, paragraph in enumerate(document["paragraphs"]):
        rows.append({
            "document_title": document["title"],
            "paragraph_index": paragraph_index,
            "paragraph_context": paragraph["context"],
        })

parsed = pd.DataFrame(
    rows,
    columns=["document_title", "paragraph_index", "paragraph_context"],
)

parsed.to_csv("parsed.tsv", sep="\t", index=False)
parsed.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [26]:
# YOUR CHANGES HERE

from sklearn.feature_extraction.text import HashingVectorizer

# Keep this vectorizer for encoding questions in Part 4.
vectorizer = HashingVectorizer(
    n_features=1024,
    stop_words="english",
    alternate_sign=False,
    norm="l2",
)

# Encode the paragraph text.
paragraph_vectors = vectorizer.transform(
    parsed["paragraph_context"]
)

# Store each vector as a JSON list alongside its paragraph identifiers.
paragraph_output = parsed[
    ["document_title", "paragraph_index"]
].copy()

paragraph_output["paragraph_vector_json"] = [
    json.dumps(paragraph_vectors[i].toarray().ravel().tolist())
    for i in range(paragraph_vectors.shape[0])
]

# The .gz extension makes pandas gzip-compress the file.
paragraph_output.to_csv(
    "paragraph-vectors.tsv.gz",
    sep="\t",
    index=False,
)

print("Vector matrix shape:", paragraph_vectors.shape)
paragraph_output.head()

Vector matrix shape: (18896, 1024)


,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,University_of_Notre_Dame,1,"[0.0, 0.0, 0.27472112789737807, 0.0, 0.0, 0.0,..."
2,University_of_Notre_Dame,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,University_of_Notre_Dame,3,"[0.0, 0.0, 0.0, 0.0, 0.6761234037828132, 0.0, ..."
4,University_of_Notre_Dame,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [27]:
# YOUR CHANGES HERE

questions = pd.read_csv("questions.tsv", sep="\t")

question_vectors = vectorizer.transform(questions["question"])

print(question_vectors.shape)

(100, 1024)


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [28]:
# YOUR CHANGES HERE

question_output = questions[["question_id"]].copy()

question_output["question_vector_json"] = [
    json.dumps(question_vectors[i].toarray().ravel().tolist())
    for i in range(question_vectors.shape[0])
]

question_output.to_csv(
    "question-vectors.tsv",
    sep="\t",
    index=False,
)

question_output.head()

,question_id,question_vector_json
0,1,"[0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,13,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [29]:
# YOUR CHANGES HERE

from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=5,
    metric="euclidean",
    algorithm="brute",
)

knn.fit(paragraph_vectors)

distances, indices = knn.kneighbors(question_vectors)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [30]:
# YOUR CHANGES HERE

matches = []

for question_position, paragraph_positions in enumerate(indices):
    question_id = questions.iloc[question_position]["question_id"]

    for rank, paragraph_position in enumerate(paragraph_positions, start=1):
        paragraph = parsed.iloc[paragraph_position]

        matches.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": paragraph["document_title"],
            "paragraph_index": paragraph["paragraph_index"],
        })

question_matches = pd.DataFrame(matches)

question_matches.to_csv(
    "question-matches.tsv",
    sep="\t",
    index=False,
)

question_matches.head(10)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Cork_(city),8
1,1,2,Southern_Europe,4
2,1,3,Geography_of_the_United_States,11
3,1,4,Russian_Soviet_Federative_Socialist_Republic,6
4,1,5,Christian,21
5,4,1,BeiDou_Navigation_Satellite_System,18
6,4,2,BeiDou_Navigation_Satellite_System,13
7,4,3,BeiDou_Navigation_Satellite_System,12
8,4,4,BeiDou_Navigation_Satellite_System,7
9,4,5,Nintendo_Entertainment_System,37


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


In [33]:
def show_matches(question_id):
    question = questions.loc[
        questions["question_id"] == question_id, "question"
    ].iloc[0]

    print(f"QUESTION {question_id}: {question}\n")

    matches = question_matches[
        question_matches["question_id"] == question_id
    ].sort_values("question_rank")

    for _, match in matches.iterrows():
        paragraph_text = parsed.loc[
            (parsed["document_title"] == match["document_title"])
            & (parsed["paragraph_index"] == match["paragraph_index"]),
            "paragraph_context",
        ].iloc[0]

        print(
            f"\nRank {match['question_rank']} | "
            f"{match['document_title']} | "
            f"Paragraph {match['paragraph_index']}"
        )
        print(paragraph_text)

In [34]:
show_matches(1)

QUESTION 1: What was the goal of the abuse of region project?


Rank 1 | Cork_(city) | Paragraph 8
The Cork accent, part of the Southwest dialect of Hiberno-English, displays various features which set it apart from other accents in Ireland. Patterns of tone and intonation often rise and fall, with the overall tone tending to be more high-pitched than other Irish accents. English spoken in Cork has a number of dialect words that are peculiar to the city and environs. Like standard Hiberno-English, some of these words originate from the Irish language, but others through other languages Cork's inhabitants encountered at home and abroad. The Cork accent displays varying degrees of rhoticity, usually depending on the social-class of the speaker.

Rank 2 | Southern_Europe | Paragraph 4
Southern Europe's flora is that of the Mediterranean Region, one of the phytochoria recognized by Armen Takhtajan. The Mediterranean and Submediterranean climate regions in Europe are found in much of Southe

In [38]:
show_matches(13)

QUESTION 13: What area is considered to have a desert climate despite having an annual monsoon season?


Rank 1 | Mali | Paragraph 10
Mali lies in the torrid zone and is among the hottest countries in the world. The thermal equator, which matches the hottest spots year-round on the planet based on the mean daily annual temperature, crosses the country. Most of Mali receives negligible rainfall and droughts are very frequent. Late June to early December is the rainy season in the southernmost area. During this time, flooding of the Niger River is common, creating the Inner Niger Delta. The vast northern desert part of Mali has a hot desert climate (Köppen climate classification (BWh) with long, extremely hot summers and scarce rainfall which decreases northwards. The central area has a hot semi-arid climate (Köppen climate classification (BSh) with very high temperatures year-round, a long, intense dry season and a brief, irregular rainy season. The little southern band possesses a trop

In [37]:
show_matches(10)

QUESTION 10: With which goddess did Sulla, Pompey, and Julius Caesar all claim a special relationship?


Rank 1 | Roman_Republic | Paragraph 47
Clodius formed armed gangs that terrorised the city and eventually began to attack Pompey's followers, who in response funded counter-gangs formed by Titus Annius Milo. The political alliance of the triumvirate was crumbling. Domitius Ahenobarbus ran for the consulship in 55 BC promising to take Caesar's command from him. Eventually, the triumvirate was renewed at Lucca. Pompey and Crassus were promised the consulship in 55 BC, and Caesar's term as governor was extended for five years. Crassus led an ill-fated expedition with legions led by his son, Caesar's lieutenant, against the Kingdom of Parthia. This resulted in his defeat and death at the Battle of Carrhae. Finally, Pompey's wife, Julia, who was Caesar's daughter, died in childbirth. This event severed the last remaining bond between Pompey and Caesar.

Rank 2 | Roman_Republic | Paragrap

In [36]:
show_matches(7)

QUESTION 7: When did Beyoncé  receive ten nominations for the Grammy Awards?


Rank 1 | Beyoncé | Paragraph 16
At the 52nd Annual Grammy Awards, Beyoncé received ten nominations, including Album of the Year for I Am... Sasha Fierce, Record of the Year for "Halo", and Song of the Year for "Single Ladies (Put a Ring on It)", among others. She tied with Lauryn Hill for most Grammy nominations in a single year by a female artist. In 2010, Beyoncé was featured on Lady Gaga's single "Telephone" and its music video. The song topped the US Pop Songs chart, becoming the sixth number-one for both Beyoncé and Gaga, tying them with Mariah Carey for most number-ones since the Nielsen Top 40 airplay chart launched in 1992. "Telephone" received a Grammy Award nomination for Best Pop Collaboration with Vocals.

Rank 2 | Beyoncé | Paragraph 54
Beyoncé has won 20 Grammy Awards, both as a solo artist and member of Destiny's Child, making her the second most honored female artist by the Grammys, behind Al

In [35]:
show_matches(4)

QUESTION 4: How many satellites in the Beidou-1 constellation?


Rank 1 | BeiDou_Navigation_Satellite_System | Paragraph 18
BeiDou-2 (formerly known as COMPASS) is not an extension to the older BeiDou-1, but rather supersedes it outright. The new system will be a constellation of 35 satellites, which include 5 geostationary orbit satellites for backward compatibility with BeiDou-1, and 30 non-geostationary satellites (27 in medium Earth orbit and 3 in inclined geosynchronous orbit), that will offer complete coverage of the globe.

Rank 2 | BeiDou_Navigation_Satellite_System | Paragraph 13
Unlike the American GPS, Russian GLONASS, and European Galileo systems, which use medium Earth orbit satellites, BeiDou-1 uses satellites in geostationary orbit. This means that the system does not require a large constellation of satellites, but it also limits the coverage to areas on Earth where the satellites are visible. The area that can be serviced is from longitude 70°E to 140°E and from latitu

In [39]:
worst_choices = {1: 8, 4: 37, 7: 14, 10: 26, 13: 27}

worst_rows = []

for question_id, paragraph_index in worst_choices.items():
    selected = question_matches.loc[
        (question_matches["question_id"] == question_id)
        & (question_matches["paragraph_index"] == paragraph_index),
        ["question_id", "document_title", "paragraph_index"],
    ]

    if len(selected) != 1:
        raise ValueError(
            f"Question {question_id}: found {len(selected)} matches "
            f"with paragraph index {paragraph_index}. "
            "Check the paragraph index and document title."
        )

    worst_rows.append(selected)

worst_paragraphs = pd.concat(worst_rows, ignore_index=True)

worst_paragraphs.to_csv(
    "worst-paragraphs.tsv",
    sep="\t",
    index=False,
)

worst_paragraphs

,question_id,document_title,paragraph_index
0,1,Cork_(city),8
1,4,Nintendo_Entertainment_System,37
2,7,Idealism,14
3,10,Roman_Republic,26
4,13,Premier_League,27


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.